# 02 - Baseline NNUE Training

Train a standard NNUE (marlinflow-style NnBoard768) as the control baseline.

**Architecture**: `768 -> ft(128) -> CReLU -> concat(stm, nstm) -> out(256 -> 1) -> sigmoid`

**Checkpoints and logs saved to Google Drive** so we can resume across sessions.

---

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q git+https://github.com/y0sif/kanue.git
!pip install -q python-chess tqdm

In [ ]:
import torch
import numpy as np
from pathlib import Path
from torch.utils.data import DataLoader

from kanue.models import NnBoard768
from kanue.models.baseline import NnBoard768Dense
from kanue.data import ChessDataset
from kanue.utils import DriveCheckpointer, train_model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')

## 1. Load Preprocessed Data from Drive

In [ ]:
DATA_DIR = Path('/content/drive/MyDrive/kanue/data')

train_data = np.load(DATA_DIR / 'train.npz')
val_data = np.load(DATA_DIR / 'val.npz')

train_ds = ChessDataset(train_data['stm'], train_data['nstm'], train_data['targets'])
val_ds = ChessDataset(val_data['stm'], val_data['nstm'], val_data['targets'])

BATCH_SIZE = 16384
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train: {len(train_ds)} positions ({len(train_loader)} batches)')
print(f'Val:   {len(val_ds)} positions ({len(val_loader)} batches)')

## 2. Define Baseline Model

In [ ]:
HIDDEN_SIZE = 128

model = NnBoard768Dense(hidden_size=HIDDEN_SIZE).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model: NnBoard768Dense(hidden={HIDDEN_SIZE})')
print(f'Total parameters: {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')
print()
print(model)

## 3. Train

In [ ]:
EPOCHS = 50
LR = 1e-3
LR_DROP_EPOCH = 35
CHECKPOINT_EVERY = 5

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
checkpointer = DriveCheckpointer('baseline')

log = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    device=device,
    epochs=EPOCHS,
    checkpointer=checkpointer,
    checkpoint_every=CHECKPOINT_EVERY,
    lr_drop_epoch=LR_DROP_EPOCH,
)

print(f'\nBest val loss: {min(log["val_loss"]):.6f}')
print(f'Best val accuracy: {max(log["val_accuracy"]):.4f}')

## 4. Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

epochs = range(len(log['train_loss']))

axes[0].plot(epochs, log['train_loss'], label='Train')
axes[0].plot(epochs, log['val_loss'], label='Val')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].set_title('Loss')
axes[0].legend()

axes[1].plot(epochs, log['val_accuracy'])
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Validation Accuracy (Winner Prediction)')

axes[2].plot(epochs, log['epoch_time'])
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Seconds')
axes[2].set_title('Epoch Time')

plt.suptitle('Baseline NNUE Training', fontweight='bold')
plt.tight_layout()

results_dir = Path('/content/drive/MyDrive/kanue/results')
plt.savefig(str(results_dir / 'baseline_training.png'), dpi=150)
plt.show()

print('Baseline training complete. Proceed to notebook 03.')